In [52]:
!pip install -q kaggle

In [53]:
import os
import zipfile

In [54]:
# Download the dataset from kaggle

os.environ["KAGGLE_USERNAME"] = "dorcaskagwiria"
os.environ["KAGGLE_KEY"] = "KGAT_148c83ee7198284e80213f155a253a7c"

!kaggle datasets download -d yashpwrr/resume-ner-training-dataset

with zipfile.ZipFile('resume-ner-training-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('resume_ner_training_dataset')

print("Dataset downloaded and unzipped")

Dataset URL: https://www.kaggle.com/datasets/yashpwrr/resume-ner-training-dataset
License(s): MIT
resume-ner-training-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset downloaded and unzipped


In [55]:
import json
import re
import random
import spacy
from spacy.tokens import DocBin

In [56]:
# Open the file in read mode
with open('resume_ner_training_dataset/train.json', 'r') as file:
    dataset = json.load(file)

In [57]:
# 1. Load a blank English spaCy model for tokenization
nlp = spacy.blank("en")

In [58]:
# def clean_text(text): #data cleaning
#     """
#     Cleans structural noise and URLs while preserving exact
#     character counts for index alignment.
#     """
#     # urls = re.findall(r'https?://\S+|www\.\S+', text)
#     # for url in urls:
#     #     text = text.replace(url, " " * len(url))
#     text = re.sub(r'[\t\r\n]', ' ', text)
#     text = re.sub(r'[●•❖➣]', ' ', text)
#     text = re.sub(r'[^a-zA-Z0-9]', ' ', text)

#     return text

# Fix unicode issues
def clean_text(text):
    return "".join(
        " " if 0xD800 <= ord(c) <= 0xDFFF else c
        for c in text
    )

In [59]:
def convert_to_docbin(dataset):
    """
    Converts annotated resume data into a spaCy DocBin.

    - Preserves text character offsets.
    - Removes leading/trailing whitespace from annotation boundaries.
    - Drops invalid and overlapping annotations.
    - Aligns annotations to spaCy token boundaries.
    - Preserves entity labels.
    """
    db = DocBin()

    for record in dataset:
        text = record.get("text", "")
        annotations = record.get("annotations", [])

        cleaned_text = clean_text(text)
        doc = nlp.make_doc(cleaned_text)

        valid_entities = []

        # Process annotations in document order
        sorted_annotations = sorted(annotations, key=lambda x: x[0])
        last_end_offset = -1

        for start, end, label in sorted_annotations:

            # 1. Validate raw boundaries
            if start < 0 or end > len(cleaned_text) or start >= end:
                continue

            # 2. Trim whitespace from annotation boundaries
            entity_text = cleaned_text[start:end]

            l_offset = len(entity_text) - len(entity_text.lstrip())
            r_offset = len(entity_text) - len(entity_text.rstrip())

            start += l_offset
            end -= r_offset

            # Ignore whitespace-only annotations
            if start >= end:
                continue

            # 3. Drop overlapping entities
            if start < last_end_offset:
                continue

            # 4. Align annotation to spaCy token boundaries
            span = doc.char_span(
                start,
                end,
                label=label.upper().strip(),
                alignment_mode="contract"
            )

            if span is None:
                continue

            # 5. Keep the ORIGINAL labeled span.
            # Do NOT reconstruct it with doc[...], because that loses the label.
            if (
                span.text.strip()
                and not span[0].is_space
                and not span[-1].is_space
            ):
              valid_entities.append(span)
              last_end_offset = span.end_char

        # Assign entities to the document
        doc.ents = valid_entities

        # Add document to DocBin
        db.add(doc)

    return db

In [60]:
# --------------------------------------------------
# 1. Shuffle the dataset
# --------------------------------------------------

random.seed(42)
random.shuffle(dataset)


# --------------------------------------------------
# 2. Calculate split sizes
# --------------------------------------------------

total = len(dataset)

train_size = int(0.80 * total)
val_size = int(0.10 * total)

train_data = dataset[:train_size]
val_data = dataset[train_size:train_size + val_size]
test_data = dataset[train_size + val_size:]


# --------------------------------------------------
# 3. Convert each split into a DocBin
# --------------------------------------------------

train_db = convert_to_docbin(train_data)
val_db = convert_to_docbin(val_data)
test_db = convert_to_docbin(test_data)


# --------------------------------------------------
# 4. Save the DocBins
# --------------------------------------------------

train_db.to_disk("train.spacy")
val_db.to_disk("validation.spacy")
test_db.to_disk("test.spacy")


# --------------------------------------------------
# 5. Check the sizes
# --------------------------------------------------

print(f"Total records:      {total}")
print(f"Training records:   {len(train_data)}")
print(f"Validation records: {len(val_data)}")
print(f"Test records:       {len(test_data)}")

Total records:      5960
Training records:   4768
Validation records: 596
Test records:       596


In [61]:
!python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency

⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [62]:
!python -m spacy debug data config.cfg --paths.train ./train.spacy --paths.dev ./validation.spacy


============================ Data file validation ============================
✔ Pipeline can be initialized with data
✔ Corpus is loadable

=============================== Training stats ===============================
Language: en
Training pipeline: tok2vec, ner
4768 training docs
596 evaluation docs
⚠ 219 training examples also in evaluation data

============================== Vocab & Vectors ==============================
ℹ 3871162 total word(s) in the data (152367 unique)
ℹ No word vectors present in the package

========================== Named Entity Recognition ==========================
ℹ 14 label(s)
0 missing value(s) (tokens with '-' label)
✔ Good amount of examples for all labels
✔ Examples without occurrences available for all labels
✔ No entities consisting of or starting/ending with whitespace
✔ No entities crossing sentence boundaries

================================== Summary ==================================
✔ 6 checks passed
⚠ 1 warning


In [66]:
!python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./validation.spacy

ℹ Saving to output directory: output
ℹ Using CPU
ℹ To switch to GPU 0, use the option: --gpu-id 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00    834.46    0.00   14.29    0.00    0.00
  0     200       2383.63  29955.29   43.79   49.71   39.14    0.44
  0     400      17327.67  22898.65   49.65   50.18   49.13    0.50
  0     600       9537.74  22080.16   54.22   49.57   59.83    0.54
  0     800       1782.35  19335.39   54.28   50.40   58.80    0.54
  0    1000      41838.15  20922.34   54.26   41.36   78.84    0.54
  0    1200      20733.95  19009.45   50.75   58.15   45.03    0.51
  0    1400      55789.40  18408.67   50.61   58.4

In [67]:
!zip -r model-best.zip ./output/model-best

updating: output/model-best/ (stored 0%)
updating: output/model-best/meta.json (deflated 68%)
updating: output/model-best/config.cfg (deflated 61%)
updating: output/model-best/ner/ (stored 0%)
updating: output/model-best/ner/model (deflated 7%)
updating: output/model-best/ner/cfg (deflated 33%)
updating: output/model-best/ner/moves (deflated 76%)
updating: output/model-best/tok2vec/ (stored 0%)
updating: output/model-best/tok2vec/model (deflated 7%)
updating: output/model-best/tok2vec/cfg (stored 0%)
updating: output/model-best/tokenizer (deflated 81%)
updating: output/model-best/vocab/ (stored 0%)
updating: output/model-best/vocab/strings.json (deflated 75%)
updating: output/model-best/vocab/vectors (deflated 45%)
updating: output/model-best/vocab/lookups.bin (stored 0%)
updating: output/model-best/vocab/key2row (stored 0%)
updating: output/model-best/vocab/vectors.cfg (stored 0%)


In [68]:
from google.colab import files
files.download("model-best.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>